# Closed fragmentation, conservative engine, constant kernel

Rebuilt from `runs/closed_fragmentation_conservative_constant.npz`. The drift is read in $t_*-t$.

## Loading

The engine and the whole measurement layer come from `BF_analysis.py` in this folder, so nothing below is defined twice. Every figure is rebuilt from the stored run; no simulation is repeated.

In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt

import BF_analysis as AN          # the whole measurement layer, engine included

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.figsize": (9, 3.2)})

FILES = {
    "closed frag": "runs/closed_fragmentation_conservative_constant.npz"
}

RUNS = {}
for tag, fn in FILES.items():
    if not os.path.exists(fn):
        raise FileNotFoundError(
            "%s is missing -- run the corresponding cell in the run-notebook next to "
            "this one; add_last_run writes it there on success." % fn)
    RUNS[tag] = AN.load(fn)
    print("loaded %-52s stop = %s" % (fn, RUNS[tag]["meta"].get("stop_reason")))

# AN.spectrum picks the estimator from meta: the averaged post-gate snapshot for an
# open run, the dt-weighted superposition for a closed one.  Both come back with an
# error bar attached.
SPEC = {tag: AN.spectrum(r) for tag, r in RUNS.items()}


# The growth-law plateau is computed here, once, from ALL the isochrones exactly as
# the engine recorded them.  Everything below reads it: the window that gets printed,
# the isochrones that get drawn and the exponent that gets quoted are then the same
# window by construction instead of by coincidence.  growth_compare also does the
# rebinning and refits it, so the check at the bottom of the notebook costs nothing
# extra here.
GROW = {tag: AN.growth_compare(r) for tag, r in RUNS.items() if "iso_counts" in r}


## What is in the file

Parameters, cost, the analysis stored at save time, and how large the counting error actually is — the empirical scatter beside the Poisson floor it can never go below.

In [ ]:
for tag, r in RUNS.items():
    print(AN.describe(r, tag))
    s = SPEC[tag]
    k = np.isfinite(s["F"]) & (s["F"] > 0)
    print("  %-16s %d snapshots averaged, K_eff median %.0f"
          % ("statistics", s["K"], np.nanmedian(s["K_eff"])))
    print("  %-16s empirical %.4f   Poisson floor %.4f   ratio %.2f"
          % ("median rel. error", np.nanmedian((s["sigma_emp"]/s["F"])[k]),
             np.nanmedian((s["sigma_pois"]/s["F"])[k]),
             np.nanmedian((s["sigma_emp"]/s["F"])[k])
             / max(np.nanmedian((s["sigma_pois"]/s["F"])[k]), 1e-30)))


## Evolution

Drift of the characteristic mass. On linear axes it must be a straight line; if $\\ln m_0$ were linear in $t$ instead, the clock would be wrong and every index would collapse onto $-2$.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
for i, (tag, r) in enumerate(RUNS.items()):
    t, m0 = np.asarray(r["t"]), np.asarray(r["m_mean"])
    a = r["meta"].get("analysis", {})
    ax[0].plot(t, m0, ".", ms=3, color="C%d" % i, label=tag)
    ts = a.get("t_star")
    x = (ts - t) if (ts and np.isfinite(ts)) else t
    sel = np.isfinite(x) & (x > 0) & (m0 > 3)
    if sel.sum() > 3:
        ax[1].loglog(x[sel], m0[sel], ".", ms=3, color="C%d" % i, label=tag)
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$m_0 = M/N$")
ax[0].set_title("(a) drift on LINEAR axes -- must be a straight line")
ax[1].set_xlabel(r"$t$  or  $t_*-t$"); ax[1].set_ylabel(r"$m_0$")
ax[1].set_title("(b) the same, log-log")
for a_ in ax: a_.legend(fontsize=7)
fig.tight_layout()

for tag, r in RUNS.items():
    M = np.asarray(r["M_sys"])
    print("%-14s m0 %.4g -> %.4g | live %d -> %d | mass %.8g -> %.8g | drift %+.1e"
          % (tag, r["m_mean"][0], r["m_mean"][-1], r["live"][0], r["live"][-1],
             M[0], M[-1], float(r["mass_drift"])))


## Where the growth law is a growth law

Every age bin is first turned into $\langle m\rangle(\tau)$ by the estimator that was
already here, untouched, and only then is the window chosen. The window is the plateau
of $d\log\langle m\rangle/d\log\tau$: the stretch of ages over which the isochrones are
self-similar, found automatically rather than set by hand. Its boundaries are printed
below before any picture is drawn, because a window that has to be seen to be believed
is not a measurement.

That same window then decides which isochrones appear on the spectrum panel. Age bins
below it are younger than one collision time and have not moved off the injection mass
— dozens of identical curves stacked in one place. Age bins above it are inside the
sink truncation. Neither is part of the cascade, and neither is drawn in colour.

Adjacent bins inside the window are merged until each curve carries enough counts to
be a curve. Merging is exact: the engine accumulates the age–mass histogram additively,
so summing neighbouring bins is precisely the histogram a coarser `iso_age_edges` would
have produced. Nothing is smoothed away that was ever measured; only age resolution is
spent, and it is spent at the top of the cascade where a single bin held a handful of
particles.

In [ ]:
# The window, stated as numbers, before it is used for anything.

for tag, r in RUNS.items():
    cmp = GROW.get(tag)
    if cmp is None:
        print("%-14s no isochrones stored in this run" % tag)
        continue
    b_th = r["meta"].get("analysis", {}).get("b_theory")
    cnt = np.asarray(r["iso_counts"]).sum(axis=1)
    print("=" * 88)
    print(AN.plateau_line(cmp["pl"], cmp["fit"], b_theory=b_th, tag=tag))
    g = cmp["groups"]
    isog = AN.iso_mean_mass(r, groups=g)
    print("  %d of %d age bins hold counts; %d of those are on the plateau,"
          " merged into %d isochrones"
          % (int((cnt > 0).sum()), cnt.size, int(cmp["k"].sum()), len(g)))
    print("  %-12s %12s %12s %13s %13s"
          % ("age bins", "tau_lo", "tau_hi", "<m>", "counts"))
    for grp, t0, t1, mb, S in zip(g, isog["tau_lo"], isog["tau_hi"],
                                  isog["mbar"], isog["counts"]):
        print("  %-12s %12.4g %12.4g %13.4g %13.4g"
              % ("%d-%d" % (grp[0], grp[-1]), t0, t1, mb, S))


## Spectrum

Compensated as $m^2\\,dN/dm$, with error bars, the plateau model and the theory line anchored to the same window so they differ in slope alone.

In [ ]:
N_CURVES = 8      # <<-- target number of merged isochrones across the plateau

n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(5.6 * n, 3.4), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a, s = axs[0][i], SPEC[tag]
    c = np.asarray(r["centers"])

    # The isochrones are chosen by the growth-law plateau printed above, not by a
    # count threshold, and adjacent age bins are merged until each curve has
    # something in it.  Everything rejected is still drawn, in grey underneath: the
    # window has to be visible against what it excludes, or the picture would be
    # confirming the window with the window.
    cmp = GROW.get(tag)
    idx, tau, ncand = AN.pick_isochrones(r, N_CURVES,
                                         pl=(cmp["pl"] if cmp else None))
    print("%-14s isochrones: %d age bins on the plateau out of %d in the grid"
          " -> %d merged curves" % (tag, ncand, len(tau), len(idx)))
    AN.draw_isochrones(a, r, idx, tau)
    if len(idx):
        tot = np.asarray(r["iso_dndm"]).sum(axis=0) / max(float(r["iso_snapshots"]), 1.0)
        a.loglog(c, np.where(tot > 0, tot * c**2, np.nan), "-", lw=3.5, alpha=.35,
                 color="0.4", zorder=2, label="sum of ALL isochrones")

    pl = AN.spectrum_plateau(r, spec=s)
    AN.plot_spectrum(a, r, spec=s, plateau=pl, label="measured $\\pm\\sigma$")

    at = r["meta"].get("analysis", {}).get("alpha_theory")
    if at and np.isfinite(pl["m_lo"]):
        xs = np.logspace(np.log10(pl["m_lo"]), np.log10(pl["m_hi"]), 30)
        A = AN.anchor_amplitude(c, s["F"], pl["m_lo"], pl["m_hi"], at)
        a.plot(xs, A * xs**at * xs**2, "--", lw=1.4, color="C1", zorder=5,
               label=r"theory $%.2f$" % at)

    w = AN.wls_powerlaw(c, s["F"], s["sigma"], mask=(c >= pl["m_lo"]) & (c <= pl["m_hi"]))
    print("      plateau alpha = %+.3f +- %.3f over %.2f dec | weighted fit %+.4f +- %.4f,"
          " chi2/dof = %.1f" % (pl["alpha"], pl["scatter"], pl["decades"],
                                w["p"], w["sigma_p"], w["chi2_dof"]))
    AN.compensated_ylim(a, c, s["F"])
    a.legend(fontsize=6, loc="lower left"); a.set_title(tag)
fig.tight_layout()
print("\nThe sum of the isochrones must reproduce the steady state -- that is the")
print("completeness check, and it uses every age bin, not the drawn subset.")
print("chi2/dof far above 1 means the deviations across the window are systematic,")
print("not statistical: the quoted sigma is then a lower bound, not the uncertainty.")


## Local slope

The diagnostic that decides the fitting window. A genuine inertial range is a plateau in $\\Gamma$; the contaminated ends are where it bends. The two shaded bands must overlap.

In [ ]:
HALF = 3        # half-width of the sliding window, in bins

n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(5.2 * n, 2.9), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a, s = axs[0][i], SPEC[tag]
    c = np.asarray(r["centers"])
    an = r["meta"].get("analysis", {})
    mm, G = AN.local_slope(c, s["F"], half=HALF)
    a.semilogx(mm, G, "o-", ms=3, lw=.8)
    if an.get("alpha_theory") is not None:
        a.axhline(an["alpha_theory"], ls="--", lw=1, color="k",
                  label=r"theory $%.2f$" % an["alpha_theory"])
    if an.get("guard_lo") and np.isfinite(an["guard_lo"]):
        a.axvspan(an["guard_lo"], an["guard_hi"], alpha=.12, label="guard band")
    pl = AN.spectrum_plateau(r, spec=s)
    if np.isfinite(pl["m_lo"]):
        a.axvspan(pl["m_lo"], pl["m_hi"], alpha=.18, color="C1", label="auto plateau")
    a.set_ylim(-4, 1); a.set_xlabel("m")
    a.set_ylabel(r"$\Gamma = d\log F/d\log m$")
    a.legend(fontsize=6); a.set_title("local slope -- %s" % tag)
fig.tight_layout()


## Mass budget

Where the mass sits and whether it balances. Closed: $M_{\\rm sys}$ constant to machine precision. Open: $M_{\\rm out}/M_{\\rm in}\\to1$ is the definition of the steady state.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
for i, (tag, r) in enumerate(RUNS.items()):
    t = np.asarray(r["t"])
    ax[0].plot(t, r["M_sys"], "-", lw=1.2, color="C%d" % i, label="%s: in the box" % tag)
    ax[0].plot(t, r["M_in"], "--", lw=1, color="C%d" % i, label="%s: injected" % tag)
    ax[0].plot(t, r["M_out"], ":", lw=1.4, color="C%d" % i, label="%s: absorbed" % tag)
    denom = np.where(np.asarray(r["M_in"]) > 0, r["M_in"], np.nan)
    ax[1].plot(t, np.asarray(r["M_out"]) / denom, "-", lw=1.2, color="C%d" % i, label=tag)
ax[0].set_xlabel("t"); ax[0].set_ylabel("mass"); ax[0].legend(fontsize=6)
ax[0].set_title("(a) where the mass is")
ax[1].axhline(1.0, ls="--", lw=1, color="k")
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$M_{\rm out}/M_{\rm in}$"); ax[1].legend(fontsize=7)
ax[1].set_title("(b) steady state means this reaches 1")
fig.tight_layout()

for tag, r in RUNS.items():
    print("%-14s M_sys %.6g -> %.6g | M_in %.4g | M_out %.4g | drift %+.2e"
          % (tag, r["M_sys"][0], r["M_sys"][-1], r["M_in"][-1], r["M_out"][-1],
             float(r["mass_drift"])))


## Evolution of the spectrum

Selected snapshots, compensated.

In [ ]:
N_SNAP = 8        # <<-- how many snapshots to draw per run

n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(5.4 * n, 3.4), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a = axs[0][i]
    c = np.asarray(r["centers"]); D = np.asarray(r["dndm"]); t = np.asarray(r["t"])
    keep = [k for k in range(D.shape[0]) if np.any(D[k] > 0)]
    sel = np.unique(np.linspace(0, len(keep) - 1, min(N_SNAP, len(keep))).astype(int))
    cm = plt.cm.plasma(np.linspace(0, .88, len(sel)))
    for col, sidx in zip(cm, sel):
        kk = keep[sidx]
        a.loglog(c, np.where(D[kk] > 0, D[kk] * c**2, np.nan), lw=1.1, color=col,
                 label="t = %.3g" % t[kk])
    a.set_xlabel("m"); a.set_ylabel(r"$m^2\,dN/dm$")
    a.legend(fontsize=5.5, ncol=2); a.set_title("evolution -- %s" % tag)
fig.tight_layout()


## The same growth law, sampled twice

The exponent is measured again on the merged age bins the isochrone panel was drawn
from, and plotted on top of the original fine-binned one. This is the check that the
widening is honest rather than flattering.

Counts add exactly, so a merged bin is the bin a coarser age grid would have recorded.
If the window really is a single power law, resampling it in age cannot move the
slope, and the two fits must agree. Where they do not, the disagreement is information:
the wide bins are straddling curvature, which means the window is not a power law over
its whole length whatever the fitted $\sigma_b$ says. Read the difference together with
$\chi^2/\rm dof$ — they fail together, and for the same reason.

In [ ]:
n = len(GROW)
if n == 0:
    print("no isochrones in this run -- nothing to compare")
else:
    fig, axs = plt.subplots(n, 2, figsize=(10.5, 3.4 * n), squeeze=False)
    for i, (tag, cmp) in enumerate(GROW.items()):
        r = RUNS[tag]
        b_th = r["meta"].get("analysis", {}).get("b_theory")
        AN.plot_growth_compare(axs[i], r, cmp=cmp, b_theory=b_th,
                               color="C%d" % i, tag=tag)
        f, fg = cmp["fit"], cmp["fit_g"]
        d = f["p"] - fg["p"]
        sd = np.hypot(f["sigma_p"], fg["sigma_p"])
        print("=" * 88)
        print("%s   theory b = %s" % (tag, b_th))
        print("  original bins   b = %+.4f +- %.4f   chi2/dof = %8.2f   n = %2d"
              % (f["p"], f["sigma_p"], f["chi2_dof"], f["n"]))
        print("  rebinned        b = %+.4f +- %.4f   chi2/dof = %8.2f   n = %2d"
              % (fg["p"], fg["sigma_p"], fg["chi2_dof"], fg["n"]))
        print("  difference        %+.4f   =  %.1f sigma"
              % (d, abs(d) / sd if sd > 0 else np.nan))
        if sd > 0 and abs(d) / sd > 3:
            print("  -> the merging is exact, so this is curvature inside the window,")
            print("     not an artefact of widening the bins.  Narrow the window or")
            print("     accept that b is a local slope here rather than an exponent.")
        else:
            print("  -> consistent: resampling the window in age does not move the slope,")
            print("     which is what a genuine power law is supposed to do.")
    fig.tight_layout()
